In [ ]:
!mkdir dataset
%cd dataset
# !gdown # mibench
!tar -xvf programs.tar.gz

In [ ]:
import os
import shutil
from glob import glob
from pathlib import Path
from tqdm import tqdm
import subprocess

# Paths
SRC_ROOT = "/content/dataset/ProgramData"

DST_ROOT = "/content/tsvb_dataset_ll"

# Step 1: Create destination directory
os.makedirs(DST_ROOT, exist_ok=True)

# Step 2: Gather all .txt files
train = 1000
txt_files = glob(f"{SRC_ROOT}/**/*.txt", recursive=True)[:train]

print(f"Found {len(txt_files)} files. Starting compilation...")

for path in tqdm(txt_files):
    try:
        # Your existing processing code here
        name = Path(path).stem
        c_path = f"/tmp/{name}.c"
        ll_path = f"{DST_ROOT}/{name}.ll"

        shutil.copy(path, c_path)
        subprocess.run(["clang", "-S", "-emit-llvm", c_path, "-o", ll_path],
                       check=True,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
    except Exception as e:
        print(f"❌ Failed to compile {path}: {e}")

print("\n✅ All valid .txt files converted to .ll and stored in:", DST_ROOT)

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.model_max_length)

def tokenize_dataset(example):
  text = example["prompt"] + example["completion"]

  tokens = tokenizer(
      text=text,
      padding='max_length',
      max_length= 2048
      truncation=True
    )

  tokens["labels"] = tokens["input_ids"].copy()
  tokens["labels"] = [
      label if label != tokenizer.pad_token_id else -100 for label in tokens["labels"]
  ]

  return tokens

dataset = load_dataset("json", data_files="pass_prediction.jsonl")["train"]
tokenized_dataset = dataset.map(tokenize_dataset)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig, TaskType
from transformers import DataCollatorForLanguageModeling
import torch

# 4-bit quantization config to save resource usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto",
)

model.resize_token_embeddings(len(tokenizer))
model.gradient_checkpointing_enable()

model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.4,
)

model = get_peft_model(model, lora_config)

args = TrainingArguments(
    output_dir="passlist-llm-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    logging_steps=20,
    save_steps=200,
    learning_rate=1e-4,
    num_train_epochs=45,
    save_total_limit=2,
    fp16=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=args,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
!pip install transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 29.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
from huggingface_hub import login

login("TOKEN")

In [ ]:
from transformers import AutoTokenizer
import transformers
import torch

model = "facebook/llm-compiler-7b"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

prompt = """You are an expert compiler engineer. Your task is to suggest the best LLVM optimization pass pipeline (comma-separated) for the following IR code.

Return only the optimization passes.

IR code:
%3 = alloca i32, align 4
"""

sequences = pipeline(
    prompt,
    do_sample=True,
    top_k=10,
    temperature=0.1,
    top_p=0.95,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    max_length=200,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

ValueError: Could not load model facebook/llm-compiler-7b with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.auto.modeling_tf_auto.TFAutoModelForCausalLM'>, <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/transformers/pipelines/base.py", line 292, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py", line 600, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py", line 311, in _wrapper
    return func(*args, **kwar

In [ ]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.6 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

model_id = "facebook/llm-compiler-7b"

# Quantization config for 4-bit inference (balanced)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Build generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Example LLVM IR snippet
# prompt = """You are an expert compiler engineer. Your task is to suggest the best LLVM optimization pass pipeline (comma-separated) for the following IR code.

# Return only the optimization passes.

# IR code:
# %3 = alloca i32, align 4
# """

prompt = """You are an expert LLVM compiler engineer.

Given the following LLVM IR, return **only** a comma-separated list of LLVM optimization passes (e.g., `mem2reg, instcombine, dce`) that would improve performance.

Do NOT include any extra explanation or IR code — just the passes.

LLVM IR:
define i32 @main() {
entry:
  %a = alloca i32, align 4
  %b = alloca i32, align 4
  store i32 42, i32* %a, align 4
  store i32 13, i32* %b, align 4
  %av = load i32, i32* %a, align 4
  %bv = load i32, i32* %b, align 4
  %sum = add nsw i32 %av, %bv
  ret i32 %sum
}
"""
# Generate passes or response
outputs = pipe(
    prompt,
    do_sample=True,
    top_k=10,
    temperature=0.1,
    top_p=0.95,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    max_length=200
)

# Show result
print("\nResult:")
print(outputs[0]["generated_text"])

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "facebook/llm-compiler-7b"

# 4-bit quantization setup
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# You are an expert LLVM compiler engineer.

# Given the following LLVM IR, return only a comma-separated list of LLVM optimization passes (e.g., `mem2reg, instcombine, dce`) that would improve performance.

# Do NOT include any extra explanation or IR code.

# LLVM IR:

# Prompt: realistic IR + clear instruction
prompt = """ Optimize:
define i32 @main() {
entry:
  %a = alloca i32, align 4
  %b = alloca i32, align 4
  store i32 42, i32* %a, align 4
  store i32 13, i32* %b, align 4
  %av = load i32, i32* %a, align 4
  %bv = load i32, i32* %b, align 4
  %sum = add nsw i32 %av, %bv
  ret i32 %sum
}
"""

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
output_tokens = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    top_k=10,
    top_p=0.95,
    temperature=0.3,
    eos_token_id=tokenizer.eos_token_id,
)

# Decode output
output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# Show only the generated part (cut off prompt)
generated = output_text[len(prompt):].strip()
print("\n🟢 Suggested optimization passes:\n", generated)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 46.12 MiB is free. Process 125098 has 14.69 GiB memory in use. Of the allocated memory 13.33 GiB is allocated by PyTorch, and 1.24 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from textwrap import indent
import torch

MODEL_NAMES = [
    "facebook/llm-compiler-7b",
]

class LLM_Compiler:
    def __init__(self, model_name: str = "facebook/llm-compiler-7b", device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        if model_name not in MODEL_NAMES:
            raise ValueError(f"model_name must be one of {MODEL_NAMES}")
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name).to(self.device)
        self.model.eval()

    def infer(self, prompt: str, max_new_tokens: int = 50) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        text: str = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return text[len(prompt): ]

    def optimize_for_code_size(self, ir: str, max_new_tokens: int = 50) -> str:
        prompt = f"""\
[INST] Tell me how to optimize this LLVM-IR for object file size:
<code>{ir}</code> [/INST]"""

        return self.infer(prompt, max_new_tokens=max_new_tokens)

if __name__ == "__main__":
    # Demo the capabilities
    ir_count = 8
    bin_size = 65
    ir = """\
; ModuleID = '<stdin>'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"
; Function Attrs: minsize nounwind optsize uwtable
define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) #0 {
  %3 = alloca i32, align 4
  %4 = alloca i32, align 4
  store i32 %0, ptr %3, align 4, !tbaa !5
  store i32 %1, ptr %4, align 4, !tbaa !5
  %5 = load i32, ptr %3, align 4, !tbaa !5
  %6 = load i32, ptr %4, align 4, !tbaa !5
  %7 = add nsw i32 %5, %6
  ret i32 %7
}
attributes #0 = { minsize nounwind optsize uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }
!llvm.module.flags = !{!0, !1, !2, !3}
!llvm.ident = !{!4}
!0 = !{i32 1, !"wchar_size", i32 4}
!1 = !{i32 8, !"PIC Level", i32 2}
!2 = !{i32 7, !"PIE Level", i32 2}
!3 = !{i32 7, !"uwtable", i32 2}
!4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
!5 = !{!6, !6, i64 0}
!6 = !{!"int", !7, i64 0}
!7 = !{!"omnipotent char", !8, i64 0}
!8 = !{!"Simple C/C++ TBAA"}
"""

passes = "module(default<Oz>)"
max_new_tokens = 800

# Get the model
llm_compiler = LLM_Compiler()

print(f"Getting the optimal passes for code size")
print(indent(llm_compiler.optimize_for_code_size(ir, max_new_tokens), "    "))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from textwrap import indent
import torch

MODEL_NAMES = ["facebook/llm-compiler-7b"]

# 4-bit quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

class LLM_Compiler:
    def __init__(self, model_name: str = "facebook/llm-compiler-7b", device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        if model_name not in MODEL_NAMES:
            raise ValueError(f"model_name must be one of {MODEL_NAMES}")
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
        self.model.eval()

    def infer(self, prompt: str, max_new_tokens: int = 50) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        text: str = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return text[len(prompt):]

    def optimize_for_code_size(self, ir: str, max_new_tokens: int = 50) -> str:
        prompt = f"""[INST] Tell me how to optimize this LLVM-IR for object file size:
<code>{ir}</code> [/INST]"""
        return self.infer(prompt, max_new_tokens=max_new_tokens)

    def suggest_passes_for_size(self, ir: str, max_new_tokens: int = 50) -> str:
      prompt = f"""[INST] Return only a comma-separated list of LLVM optimization passes to minimize code size:
  <code>{ir}</code> [/INST]"""
      return self.infer(prompt, max_new_tokens=max_new_tokens)


# Example usage
if __name__ == "__main__":
    # Demo the capabilities
    ir_count = 8
    bin_size = 65
    ir = """\
; ModuleID = '<stdin>'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"
; Function Attrs: minsize nounwind optsize uwtable
define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) #0 {
  %3 = alloca i32, align 4
  %4 = alloca i32, align 4
  store i32 %0, ptr %3, align 4, !tbaa !5
  store i32 %1, ptr %4, align 4, !tbaa !5
  %5 = load i32, ptr %3, align 4, !tbaa !5
  %6 = load i32, ptr %4, align 4, !tbaa !5
  %7 = add nsw i32 %5, %6
  ret i32 %7
}
attributes #0 = { minsize nounwind optsize uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }
!llvm.module.flags = !{!0, !1, !2, !3}
!llvm.ident = !{!4}
!0 = !{i32 1, !"wchar_size", i32 4}
!1 = !{i32 8, !"PIC Level", i32 2}
!2 = !{i32 7, !"PIE Level", i32 2}
!3 = !{i32 7, !"uwtable", i32 2}
!4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
!5 = !{!6, !6, i64 0}
!6 = !{!"int", !7, i64 0}
!7 = !{!"omnipotent char", !8, i64 0}
!8 = !{!"Simple C/C++ TBAA"}
"""

# second run
# ir = """\
  # ; ModuleID = '<stdin>'
  # source_filename = "-"
  # target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
  # target triple = "x86_64-unknown-linux-gnu"

  # ; Function Attrs: minsize nounwind optsize uwtable
  # define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) #0 {
  #   %3 = add nsw i32 %0, %1
  #   ret i32 %3
  # }

  # attributes #0 = { minsize nounwind optsize uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

  # !llvm.module.flags = !{!0, !1, !2, !3}
  # !llvm.ident = !{!4}

  # !0 = !{i32 1, !"wchar_size", i32 4}
  # !1 = !{i32 8, !"PIC Level", i32 2}
  # !2 = !{i32 7, !"PIE Level", i32 2}
  # !3 = !{i32 7, !"uwtable", i32 2}
  # !4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
# """

passes = "module(default<Oz>)"
max_new_tokens = 800

# Get the model
llm_compiler = LLM_Compiler()

# print(f"Getting the optimal passes for code size")
# print(indent(llm_compiler.optimize_for_code_size(ir, max_new_tokens), "    "))

print("Suggested optimization passes:")
print(indent(llm_compiler.suggest_passes_for_size(ir, max_new_tokens), "    "))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Suggested optimization passes:
      The LLVM-IR will have instruction count 2 and binary sise 54 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    ; Function Attrs: minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable
    define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) local_unnamed_addr #0 {
      %3 = add nsw i32 %1, %0
      ret i32 %3
    }

    attributes #0 = { minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

    !llvm.module.flags = !{!0, !1, !2, !3}
    !llvm.ident = !{!4}

    !0 = !{i32 1, !"wchar_size", i32 4}
    !1 = !{i3

Best Run:


Loading checkpoint shards: 100%
 3/3 [01:11<00:00, 22.48s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Getting the optimal passes for code size
      The LLVM-IR will have instruction count 3 and binary sise 57 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    ; Function Attrs: minsize nounwind optsize uwtable
    define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) #0 {
      %3 = add nsw i32 %0, %1
      ret i32 %3
    }

    attributes #0 = { minsize nounwind optsize uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

    !llvm.module.flags = !{!0, !1, !2, !3}
    !llvm.ident = !{!4}

    !0 = !{i32 1, !"wchar_size", i32 4}
    !1 = !{i32 8, !"PIC Level", i32 2}
    !2 = !{i32 7, !"PIE Level", i32 2}
    !3 = !{i32 7, !"uwtable", i32 2}
    !4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
    </code>
  
  Second Run:
  Loading checkpoint shards: 100%
 3/3 [01:06<00:00, 21.23s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Getting the optimal passes for code size
      The LLVM-IR will have instruction count 3 and binary sise 57 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    ; Function Attrs: minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable
    define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) local_unnamed_addr #0 {
      %3 = add nsw i32 %1, %0
      ret i32 %3
    }

    attributes #0 = { minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

    !llvm.module.flags = !{!0, !1, !2, !3}
    !llvm.ident = !{!4}

    !0 = !{i32 1, !"wchar_size", i32 4}
    !1 = !{i32 8, !"PIC Level", i32 2}
    !2 = !{i32 7, !"PIE Level", i32 2}
    !3 = !{i32 7, !"uwtable", i32 2}
    !4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
    </code>

Trying to get an opt pass list but instead reduced code:

Loading checkpoint shards: 100%
 3/3 [01:09<00:00, 22.10s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Suggested optimization passes:
      The LLVM-IR will have instruction count 2 and binary sise 54 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    ; Function Attrs: minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable
    define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) local_unnamed_addr #0 {
      %3 = add nsw i32 %1, %0
      ret i32 %3
    }

    attributes #0 = { minsize mustprogress nofree norecurse nosync nounwind optsize willreturn memory(none) uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

    !llvm.module.flags = !{!0, !1, !2, !3}
    !llvm.ident = !{!4}

    !0 = !{i32 1, !"wchar_size", i32 4}
    !1 = !{i32 8, !"PIC Level", i32 2}
    !2 = !{i32 7, !"PIE Level", i32 2}
    !3 = !{i32 7, !"uwtable", i32 2}
    !4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
    </code>

Result:
You are an expert compiler engineer. Your task is to suggest the best LLVM optimization pass pipeline (comma-separated) for the following IR code.

Return only the optimization passes.

IR code:
%3 = alloca i32, align 4
%4 = alloca i32, align 4
%5 = alloca i32, align 4
%6 = alloca i32, align 4
%7 = alloca i32, align 4
%8 = alloca i32, align 4
%9 = alloca i32, align 4
%10 = alloca i32, align 4
%11 = alloca i32, align 4
%12 = alloca i32, align 4
%13 = alloca i32, align 4
%14 = alloca i32, align 4
%15 = alloca i32, align 4
%16 = alloca i32, align 4
%17 = alloca i32, align 4
%18 = alloca i32, align 4
%19 = alloca i32, align 4
%20 = alloca i32, align 4
%21 = alloca i32, align 4
%22 = alloca i32,


🟢 Suggested optimization passes:
 !llvm.module.flags = !{!0, !1, !2, !3}
!llvm.ident = !{!4}

!0 = !{i32 1, !"wchar_size", i32 4}
!1

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Loading checkpoint shards: 100%
 3/3 [01:06<00:00, 21.19s/it]
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Getting the optimal passes for code size:
      The LLVM-IR will have instruction count 3 and binary sise 54 bytes:

    <code>; ModuleID = '<stdin>'
    source_filename = "-"
    target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
    target triple = "x86_64-unknown-linux-gnu"

    ; Function Attrs: minsize nounwind optsize uwtable
    define dso_local i32 @add_two(i32 noundef %0, i32 noundef %1) #0 {
      %3 = add nsw i32 %0, %1
      ret i32 %3
    }

    attributes #0 = { minsize nounwind optsize uwtable "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="x86-64" "target-features"="+cmov,+cx8,+fxsr,+mmx,+sse,+sse2,+x87" "tune-cpu"="generic" }

    !llvm.module.flags = !{!0, !1, !2, !3}
    !llvm.ident = !{!4}

    !0 = !{i32 1, !"wchar_size", i32 4}
    !1 = !{i32 8, !"PIC Level", i32 2}
    !2 = !{i32 7, !"PIE Level", i32 2}
    !3 = !{i32 7, !"uwtable", i32 2}
    !4 = !{!"clang version 17.0.6 (git@github.com:fairinternal/CodeGen.git b05db9bbf7a92019267416c1bb9996fe6134e3f1)"}
    </code>